# QNN Integration: Common Comparison Plots

This notebook compares the same metric across result sets produced by `ExperimentResultWriter`. It includes final summary comparisons, subset-level test metric distributions, and epoch-curve comparisons.

In [1]:
from pathlib import Path
import json
import re

from IPython.display import Markdown, display
import numpy as np
import pandas as pd
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


pio.templates["latex"] = pio.templates["plotly_white"].update(
    layout=dict(
        colorway=px.colors.qualitative.D3,
        font=dict(
            family="CMU Serif",
            size=12
        )
    )
)

pio.templates.default = "latex"

pd.options.display.max_columns = 120
pd.options.display.max_colwidth = 160

In [2]:
EPOCH_COLUMN_MAP = {
    "TrLoss": ("train", "loss"),
    "TrAcc": ("train", "accuracy"),
    "TrPrec": ("train", "precision"),
    "TrRec": ("train", "recall"),
    "TrF1": ("train", "f1"),
    "VaLoss": ("val", "loss"),
    "VaAcc": ("val", "accuracy"),
    "VaPrec": ("val", "precision"),
    "VaRec": ("val", "recall"),
    "VaF1": ("val", "f1"),
}
METRIC_ORDER = ["loss", "accuracy", "precision", "recall", "f1"]
SUBSET_RE = re.compile(r"subset_(\d+)_(epochs|test_metrics)\.csv$")


def locate_results_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend([
            base / "experiments" / "QNN_integration" / "results",
            base / "results",
        ])

    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("*/*/manifest.json")):
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not find experiments/QNN_integration/results from the current working directory."
    )


def subset_index(path: Path) -> int:
    match = SUBSET_RE.match(path.name)
    if not match:
        raise ValueError(f"Unexpected subset result filename: {path.name}")
    return int(match.group(1))


def discover_runs(results_root: Path) -> pd.DataFrame:
    records = []
    for manifest_path in sorted(results_root.glob("*/*/manifest.json")):
        run_dir = manifest_path.parent
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            manifest = {}

        pipeline = manifest.get("pipeline_name") or run_dir.parent.name
        timestamp = run_dir.name
        epoch_files = sorted(run_dir.glob("subset_*_epochs.csv"))
        metric_files = sorted(run_dir.glob("subset_*_test_metrics.csv"))
        records.append(
            {
                "run_id": f"{pipeline}/{timestamp}",
                "result_set": f"{pipeline} / {timestamp}",
                "pipeline": pipeline,
                "timestamp": timestamp,
                "run_dir": str(run_dir),
                "manifest_path": str(manifest_path),
                "started_at": manifest.get("started_at"),
                "completed_at": manifest.get("completed_at"),
                "subset_count_manifest": len(manifest.get("subsets", [])),
                "epoch_file_count": len(epoch_files),
                "test_metric_file_count": len(metric_files),
                "has_summary": (run_dir / "final_summary_across.csv").exists(),
            }
        )

    runs = pd.DataFrame.from_records(records)
    if runs.empty:
        return runs
    return runs.sort_values(["pipeline", "timestamp"]).reset_index(drop=True)


def load_epoch_history(runs: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for run in runs.itertuples(index=False):
        run_dir = Path(run.run_dir)
        for path in sorted(run_dir.glob("subset_*_epochs.csv")):
            df = pd.read_csv(path, sep=";")
            df.columns = [column.strip() for column in df.columns]
            for column in df.columns:
                df[column] = pd.to_numeric(df[column], errors="coerce")
            df["subset"] = subset_index(path)
            df["pipeline"] = run.pipeline
            df["timestamp"] = run.timestamp
            df["run_id"] = run.run_id
            df["result_set"] = run.result_set
            df["source_file"] = str(path)
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_test_metrics(runs: pd.DataFrame) -> pd.DataFrame:
    records = []
    for run in runs.itertuples(index=False):
        run_dir = Path(run.run_dir)
        for path in sorted(run_dir.glob("subset_*_test_metrics.csv")):
            metrics = pd.read_csv(path, sep=";")
            metric_row = {
                "subset": subset_index(path),
                "pipeline": run.pipeline,
                "timestamp": run.timestamp,
                "run_id": run.run_id,
                "result_set": run.result_set,
                "source_file": str(path),
            }
            for item in metrics.itertuples(index=False):
                metric_row[str(item.metric)] = pd.to_numeric(item.value, errors="coerce")
            records.append(metric_row)

    return pd.DataFrame.from_records(records)


def load_final_summaries(runs: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for run in runs.itertuples(index=False):
        path = Path(run.run_dir) / "final_summary_across.csv"
        if not path.exists():
            continue
        df = pd.read_csv(path, sep=";")
        for column in ["mean", "std"]:
            df[column] = pd.to_numeric(df[column], errors="coerce")
        df["pipeline"] = run.pipeline
        df["timestamp"] = run.timestamp
        df["run_id"] = run.run_id
        df["result_set"] = run.result_set
        df["source_file"] = str(path)
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def to_long_epoch_frame(epoch_history: pd.DataFrame) -> pd.DataFrame:
    value_columns = [column for column in EPOCH_COLUMN_MAP if column in epoch_history.columns]
    id_columns = [
        "Epoch",
        "subset",
        "pipeline",
        "timestamp",
        "run_id",
        "result_set",
        "source_file",
    ]
    long_df = epoch_history.melt(
        id_vars=id_columns,
        value_vars=value_columns,
        var_name="history_column",
        value_name="value",
    ).dropna(subset=["value"])

    column_meta = pd.DataFrame(
        [
            {"history_column": column, "split": split, "metric": metric}
            for column, (split, metric) in EPOCH_COLUMN_MAP.items()
        ]
    )
    return long_df.merge(column_meta, on="history_column", how="left")


def to_long_test_frame(test_metrics: pd.DataFrame) -> pd.DataFrame:
    value_columns = [column for column in ["loss", "accuracy", "precision", "recall", "f1"] if column in test_metrics.columns]
    id_columns = [
        "subset",
        "pipeline",
        "timestamp",
        "run_id",
        "result_set",
        "source_file",
    ]
    return test_metrics.melt(
        id_vars=id_columns,
        value_vars=value_columns,
        var_name="metric",
        value_name="value",
    ).dropna(subset=["value"])


def aggregate_with_ci(df: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    agg = (
        df.groupby(group_columns, dropna=False)["value"]
        .agg(mean="mean", std="std", count="count")
        .reset_index()
    )
    agg["std"] = agg["std"].fillna(0.0)
    agg["sem"] = np.where(agg["count"] > 0, agg["std"] / np.sqrt(agg["count"]), 0.0)
    agg["t_critical"] = agg["count"].apply(lambda n: stats.t.ppf(0.975, n - 1) if n > 1 else 0.0)
    agg["ci95"] = agg["sem"] * agg["t_critical"]
    agg["lower"] = agg["mean"] - agg["ci95"]
    agg["upper"] = agg["mean"] + agg["ci95"]
    return agg


def add_ci_band(fig: go.Figure, x, lower, upper, color: str, name: str, row=None, col=None) -> None:
    fig.add_trace(
        go.Scatter(
            x=list(x) + list(x)[::-1],
            y=list(upper) + list(lower)[::-1],
            fill="toself",
            fillcolor=color,
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            name=name,
            showlegend=False,
        ),
        row=row,
        col=col,
    )


RESULTS_ROOT = locate_results_root()
runs = discover_runs(RESULTS_ROOT)
runs_with_epochs = runs[runs["epoch_file_count"] > 0].copy()
epoch_history = load_epoch_history(runs_with_epochs)
epoch_long = to_long_epoch_frame(epoch_history) if not epoch_history.empty else pd.DataFrame()
test_metrics = load_test_metrics(runs_with_epochs)
test_long = to_long_test_frame(test_metrics) if not test_metrics.empty else pd.DataFrame()
final_summaries = load_final_summaries(runs_with_epochs)

print(f"Results root: {RESULTS_ROOT}")
print(f"Discovered {len(runs)} manifest-backed result sets, {len(runs_with_epochs)} with epoch histories.")

Results root: /home/ODM/mkordasz/geqie/experiments/QNN_integration/results
Discovered 10 manifest-backed result sets, 9 with epoch histories.


In [3]:
runs_with_epochs[[
    "result_set",
    "started_at",
    "completed_at",
    "epoch_file_count",
    "test_metric_file_count",
    "has_summary",
    "run_dir",
]]

,result_set,started_at,completed_at,epoch_file_count,test_metric_file_count,has_summary,run_dir
0,Baseline_A_Dense / 29-04-2026-13-20,2026-04-29T13:20:41,2026-04-29T13:20:48,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20
1,Baseline_A_Dense / 29-04-2026-13-22,2026-04-29T13:22:04,2026-04-29T13:22:10,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22
2,Baseline_A_Dense / 29-04-2026-13-23,2026-04-29T13:23:15,2026-04-29T13:23:20,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-23
4,Baseline_A_Dense / 29-04-2026-14-21,2026-04-29T14:21:08,2026-04-29T14:21:15,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-14-21
5,Baseline_B_CNN / 29-04-2026-14-21,2026-04-29T14:21:15,2026-04-29T14:21:24,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_B_CNN/29-04-2026-14-21
6,Baseline_C / 29-04-2026-14-23,2026-04-29T14:23:00,NaN,3,3,False,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_C/29-04-2026-14-23
7,Baseline_C / 29-04-2026-19-26,2026-04-29T19:26:10,2026-04-30T01:44:35,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_C/29-04-2026-19-26
8,Baseline_D / 30-04-2026-01-44,2026-04-30T01:44:35,2026-04-30T08:30:15,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_D/30-04-2026-01-44
9,Experiment_A_FRQI / 30-04-2026-09-18,2026-04-30T09:18:31,2026-04-30T12:29:25,5,5,True,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Experiment_A_FRQI/30-04-2026-09-18


## Select Result Sets

By default every manifest-backed run with epoch histories is included. Set `USE_LATEST_RUN_PER_PIPELINE = True` to compare only the newest timestamp under each pipeline.

In [4]:
USE_LATEST_RUN_PER_PIPELINE = False
SELECTED_RESULT_SETS = None

comparison_runs = runs_with_epochs.copy()
if USE_LATEST_RUN_PER_PIPELINE:
    comparison_runs = (
        comparison_runs.sort_values(["pipeline", "timestamp"])
        .groupby("pipeline", as_index=False)
        .tail(1)
        .sort_values(["pipeline", "timestamp"])
    )

if SELECTED_RESULT_SETS is not None:
    comparison_runs = comparison_runs[comparison_runs["result_set"].isin(SELECTED_RESULT_SETS)].copy()

comparison_result_sets = comparison_runs["result_set"].tolist()
comparison_result_sets

['Baseline_A_Dense / 29-04-2026-13-20',
 'Baseline_A_Dense / 29-04-2026-13-22',
 'Baseline_A_Dense / 29-04-2026-13-23',
 'Baseline_A_Dense / 29-04-2026-14-21',
 'Baseline_B_CNN / 29-04-2026-14-21',
 'Baseline_C / 29-04-2026-14-23',
 'Baseline_C / 29-04-2026-19-26',
 'Baseline_D / 30-04-2026-01-44',
 'Experiment_A_FRQI / 30-04-2026-09-18']

## Final Summary Comparison

In [5]:
def plot_final_summary_comparison(summary_df: pd.DataFrame) -> go.Figure:
    data = summary_df[summary_df["result_set"].isin(comparison_result_sets)].copy()
    if data.empty:
        raise ValueError("No final summary data are available for the selected result sets.")

    metrics = [metric for metric in METRIC_ORDER if metric != "loss" and metric in data["metric"].unique()]
    fig = make_subplots(
        rows=len(metrics),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.055,
        subplot_titles=[metric.title() for metric in metrics],
    )

    color_map = {
        result_set: px.colors.qualitative.D3[index % len(px.colors.qualitative.D3)]
        for index, result_set in enumerate(comparison_result_sets)
    }

    for row_number, metric in enumerate(metrics, start=1):
        metric_df = data[data["metric"] == metric].set_index("result_set").reindex(comparison_result_sets).reset_index()
        fig.add_trace(
            go.Bar(
                x=metric_df["result_set"],
                y=metric_df["mean"],
                error_y=dict(type="data", array=metric_df["std"], visible=True),
                marker_color=[color_map[name] for name in metric_df["result_set"]],
                name=metric.title(),
                showlegend=False,
                hovertemplate="%{x}<br>mean=%{y:.4f}<extra></extra>",
            ),
            row=row_number,
            col=1,
        )

    fig.update_layout(
        title="Final Summary Across Result Sets",
        height=max(360, 245 * len(metrics)),
    )
    fig.update_xaxes(tickangle=35)
    fig.update_yaxes(title_text="Mean +/- std")
    return fig


if final_summaries.empty:
    display(Markdown("No `final_summary_across.csv` files were found."))
else:
    plot_final_summary_comparison(final_summaries).show()

## Subset Test Metric Distributions

In [6]:
def plot_test_metric_distribution(metric: str) -> go.Figure:
    data = test_long[
        (test_long["result_set"].isin(comparison_result_sets))
        & (test_long["metric"] == metric)
    ].copy()
    if data.empty:
        raise ValueError(f"No subset test data found for metric {metric!r}.")

    fig = px.box(
        data,
        x="result_set",
        y="value",
        color="pipeline",
        points="all",
        hover_data=["subset"],
        title=f"Subset Test {metric.title()} Across Result Sets",
    )
    fig.update_layout(height=500, xaxis_title="Result Set", yaxis_title=metric.title())
    fig.update_xaxes(tickangle=35)
    return fig


for metric in [metric for metric in METRIC_ORDER if metric in test_long["metric"].unique()]:
    plot_test_metric_distribution(metric).show()

## Epoch Curve Comparison

In [7]:
def color_with_alpha(color: str, alpha: float) -> str:
    if color.startswith("#"):
        value = color.lstrip("#")
        if len(value) == 3:
            value = "".join(channel * 2 for channel in value)
        red, green, blue = (int(value[index:index + 2], 16) for index in (0, 2, 4))
        return f"rgba({red},{green},{blue},{alpha})"
    if color.startswith("rgb("):
        return color.replace("rgb(", "rgba(").replace(")", f",{alpha})")
    return f"rgba(0,0,0,{alpha})"


def plot_epoch_metric_comparison(metric: str, split: str = "val") -> go.Figure:
    data = epoch_long[
        (epoch_long["result_set"].isin(comparison_result_sets))
        & (epoch_long["metric"] == metric)
        & (epoch_long["split"] == split)
    ].copy()
    if data.empty:
        raise ValueError(f"No epoch data found for metric={metric!r}, split={split!r}.")

    aggregated = aggregate_with_ci(data, ["result_set", "pipeline", "Epoch", "split", "metric"])
    color_map = {
        result_set: px.colors.qualitative.D3[index % len(px.colors.qualitative.D3)]
        for index, result_set in enumerate(comparison_result_sets)
    }

    fig = go.Figure()
    for result_set in comparison_result_sets:
        result_df = aggregated[aggregated["result_set"] == result_set].sort_values("Epoch")
        if result_df.empty:
            continue
        color = color_map[result_set]
        add_ci_band(
            fig,
            result_df["Epoch"],
            result_df["lower"],
            result_df["upper"],
            color_with_alpha(color, 0.12),
            f"{result_set} 95% CI",
        )
        fig.add_trace(
            go.Scatter(
                x=result_df["Epoch"],
                y=result_df["mean"],
                mode="lines+markers",
                line=dict(color=color, width=2.5),
                marker=dict(size=5),
                name=result_set,
                hovertemplate="epoch=%{x}<br>mean=%{y:.4f}<extra></extra>",
            )
        )

    fig.update_layout(
        title=f"{split.title()} {metric.title()} Over Epochs",
        height=520,
        hovermode="x unified",
        xaxis_title="Epoch",
        yaxis_title=metric.title(),
        legend_title_text="Result Set",
    )
    return fig


COMPARE_SPLIT = "val"
for metric in [metric for metric in METRIC_ORDER if metric in epoch_long["metric"].unique()]:
    plot_epoch_metric_comparison(metric, split=COMPARE_SPLIT).show()

## Comparison Data Tables

In [8]:
summary_table = final_summaries[final_summaries["result_set"].isin(comparison_result_sets)].copy()
subset_table = test_metrics[test_metrics["result_set"].isin(comparison_result_sets)].copy()

display(Markdown("### Final summaries"))
display(summary_table.sort_values(["result_set", "metric"]))

display(Markdown("### Subset test metrics"))
display(subset_table.sort_values(["result_set", "subset"]))

### Final summaries

,metric,mean,std,pipeline,timestamp,run_id,result_set,source_file
0,accuracy,0.840972,0.014467,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/final_summary_across.csv
3,f1,0.840073,0.013894,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/final_summary_across.csv
1,precision,0.845692,0.015125,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/final_summary_across.csv
2,recall,0.840972,0.014467,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/final_summary_across.csv
4,accuracy,0.843750,0.019145,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/final_summary_across.csv
7,f1,0.842902,0.019027,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/final_summary_across.csv
5,precision,0.847490,0.020426,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/final_summary_across.csv
6,recall,0.843750,0.019145,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/final_summary_across.csv
8,accuracy,0.849306,0.017650,Baseline_A_Dense,29-04-2026-13-23,Baseline_A_Dense/29-04-2026-13-23,Baseline_A_Dense / 29-04-2026-13-23,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-23/final_summary_across.csv
11,f1,0.848001,0.017050,Baseline_A_Dense,29-04-2026-13-23,Baseline_A_Dense/29-04-2026-13-23,Baseline_A_Dense / 29-04-2026-13-23,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-23/final_summary_across.csv


### Subset test metrics

,subset,pipeline,timestamp,run_id,result_set,source_file,loss,accuracy,precision,recall,f1
0,1,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/subset_01_test_metrics.csv,0.836259,0.822917,0.830239,0.822917,0.823384
1,2,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/subset_02_test_metrics.csv,0.813757,0.829861,0.833785,0.829861,0.828506
2,3,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/subset_03_test_metrics.csv,0.835871,0.847222,0.852115,0.847222,0.847323
3,4,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/subset_04_test_metrics.csv,0.724560,0.840278,0.840302,0.840278,0.838751
4,5,Baseline_A_Dense,29-04-2026-13-20,Baseline_A_Dense/29-04-2026-13-20,Baseline_A_Dense / 29-04-2026-13-20,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-20/subset_05_test_metrics.csv,0.752013,0.864583,0.872017,0.864583,0.862402
5,1,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/subset_01_test_metrics.csv,0.838544,0.812500,0.816278,0.812500,0.812557
6,2,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/subset_02_test_metrics.csv,0.811467,0.833333,0.834955,0.833333,0.831447
7,3,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/subset_03_test_metrics.csv,0.839260,0.861111,0.865909,0.861111,0.861383
8,4,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/subset_04_test_metrics.csv,0.716170,0.847222,0.848017,0.847222,0.846056
9,5,Baseline_A_Dense,29-04-2026-13-22,Baseline_A_Dense/29-04-2026-13-22,Baseline_A_Dense / 29-04-2026-13-22,/home/ODM/mkordasz/geqie/experiments/QNN_integration/results/Baseline_A_Dense/29-04-2026-13-22/subset_05_test_metrics.csv,0.754590,0.864583,0.872293,0.864583,0.863066
